In [ ]:
DROP TABLE IF EXISTS silver_rdm_form_question_add;

CREATE TABLE silver_rdm_form_question_add AS

WITH mpb_source AS (

    SELECT DISTINCT
        CONCAT('MPB001_', CAST(asmt.id AS STRING)) AS form_ques_src_id,
        'MPB001' AS form_ques_src_sys_inst_id,
        CONCAT_WS('_', asmt.name, asmt.subdomain, asmt.scoring_method) AS form_ques_src_name
    FROM silver_drj_assessments asmt
    WHERE asmt.id IS NOT NULL

),

wip_source AS (

    SELECT DISTINCT
        'Unknown' AS form_ques_src_id,
        'Unknown' AS form_ques_src_sys_inst_id,
        'Unknown' AS form_ques_src_name
    WHERE 1 = 0

),

source_form_questions AS (

    SELECT *
    FROM mpb_source

    UNION

    SELECT *
    FROM wip_source

)

SELECT DISTINCT
    src.form_ques_src_id,
    src.form_ques_src_sys_inst_id,
    src.form_ques_src_name
FROM source_form_questions src
LEFT JOIN silver_rdm_form_question rdm
    ON src.form_ques_src_id = rdm.form_ques_src_id
   AND src.form_ques_src_sys_inst_id = rdm.form_ques_src_sys_inst_id
WHERE rdm.form_ques_src_id IS NULL;

In [ ]:
Added the MPB source block for RDM Form Question additions using silver_drj_assessments. The code has been structured with a shared source_form_questions CTE so additional source systems can be added later using separate UNION blocks before the final left join against silver_rdm_form_question.

In [ ]:
SELECT
    ty.id AS service_type_id,
    ty.description AS service_type_description,
    sg.id AS statistical_group_id,
    sg.description AS statistical_group_description,
    st.id AS statistical_type_id,
    st.description AS statistical_type_description
FROM silver_wip_statisticalgroup sg
LEFT JOIN silver_wip_statisticaltype st
    ON sg.id = st.statistical_group_id
LEFT JOIN silver_wip_statisticalchoice sc
    ON st.id = sc.statistical_type_id
LEFT JOIN silver_wip_statistic s
    ON sc.id = s.statistical_choice_id
LEFT JOIN silver_wip_activityheader ah
    ON s.activity_header_id = ah.id
LEFT JOIN silver_wip_servicetype ty
    ON ah.service_type_id = ty.id
LIMIT 50;

In [ ]:
SELECT DISTINCT
    CONCAT(
        'WIP001_',
        CAST(ty.id AS STRING),
        '_',
        CAST(sg.id AS STRING),
        '_',
        CAST(st.id AS STRING)
    ) AS form_ques_src_id,
    'WIP001' AS form_ques_src_sys_inst_id,
    CONCAT_WS(
        ' - ',
        ty.description,
        sg.description,
        st.description
    ) AS form_ques_src_name
FROM silver_wip_statisticalgroup sg
LEFT JOIN silver_wip_statisticaltype st
    ON sg.id = st.statistical_group_id
LEFT JOIN silver_wip_statisticalchoice sc
    ON st.id = sc.statistical_type_id
LEFT JOIN silver_wip_statistic s
    ON sc.id = s.statistical_choice_id
LEFT JOIN silver_wip_activityheader ah
    ON s.activity_header_id = ah.id
LEFT JOIN silver_wip_servicetype ty
    ON ah.service_type_id = ty.id
WHERE ty.id IS NOT NULL
  AND sg.id IS NOT NULL
  AND st.id IS NOT NULL
LIMIT 50;

In [ ]:
wip_source AS (

    SELECT DISTINCT
        CONCAT('WIP001_', CAST(ty.id AS STRING), '_', CAST(sg.id AS STRING), '_', CAST(st.id AS STRING)) AS form_ques_src_id,
        'WIP001' AS form_ques_src_sys_inst_id,
        CONCAT_WS(' - ', ty.description, sg.description, st.description) AS form_ques_src_name
    FROM silver_wip_statisticalgroup sg
    LEFT JOIN silver_wip_statisticaltype st
        ON sg.id = st.statistical_group_id
    LEFT JOIN silver_wip_statisticalchoice sc
        ON st.id = sc.statistical_type_id
    LEFT JOIN silver_wip_statistic s
        ON sc.id = s.statistical_choice_id
    LEFT JOIN silver_wip_activityheader ah
        ON s.activity_header_id = ah.id
    LEFT JOIN silver_wip_servicetype ty
        ON ah.service_type_id = ty.id
    WHERE ty.id IS NOT NULL AND sg.id IS NOT NULL AND st.id IS NOT NULL

)